# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates step-by-step how to explore and process the dataset using the `mlcroissant` library, referencing all entities using their `@id` fields for full reproducibility and traceability.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This step instantiates the dataset and displays its high-level metadata attributes.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL (use the provided variable)
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset schema and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` fields. This will help us decide which record sets and fields to analyze.

Below, we list the available record sets and their field `@id`s.

In [ ]:
# List all record sets and their fields by @id

recordset_ids = [r['@id'] for r in metadata.record_sets]
print("Available record sets (@id):", recordset_ids)
for record_set in metadata.record_sets:
    print(f"\nRecordSet: {record_set['@id']}")
    if 'field' in record_set:
        field_ids = [(f['@id'] if isinstance(f, dict) else f) for f in record_set['field']]
        print("  Field @ids:", field_ids)
    else:
        print("  No fields defined.")

## 3. Data Extraction
Load data from selected record sets into pandas DataFrames for analysis. Use the record set and field `@id`s from the previous section.

Below, we demonstrate loading the primary protein record set and displaying its columns.

In [ ]:
# Example: Assume the main proteomics table is cr:recordSet_1 (change if needed based on overview)
record_sets = recordset_ids  # All available record set @ids
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records, {df.shape[1]} columns.")
    print(f"Sample columns for {record_set_id}:", df.columns.tolist())

# For example, let's pick the first record set for downstream analysis
main_record_set_id = recordset_ids[0]
print(f"Using main record set: {main_record_set_id}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering out low abundance proteins, normalizing numeric fields, and grouping by annotation source. All operations reference the field using their `@id` fields, as discovered above.


In [ ]:
# Identify a numeric field to filter (e.g., cr:normalized_abundance or cr:coverage_percent)
numeric_candidates = [col for col in dataframes[main_record_set_id].columns if dataframes[main_record_set_id][col].dtype in ['float64', 'int64']]
print("Numeric field candidates:", numeric_candidates)

# Example: Use first numeric field found
numeric_field = numeric_candidates[0]
print(f"Using numeric field {numeric_field} for filtering.")

# Filter records with numeric_field > threshold
threshold = dataframes[main_record_set_id][numeric_field].quantile(0.75)
filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field] > threshold].copy()
print(f"Records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
normalized_col = f"{numeric_field}_normalized"
filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, normalized_col]].head())

# Group by annotation/protein description field if available
group_candidates = [col for col in dataframes[main_record_set_id].columns if dataframes[main_record_set_id][col].dtype == object]
print("Categorical/group field candidates:", group_candidates)
# Use the second field as a group, if available
if len(group_candidates) > 1:
    group_field = group_candidates[1]
else:
    group_field = group_candidates[0]
print(f"Grouping by: {group_field}")
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Mean {numeric_field} grouped by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
We visualize the distribution of the main numeric field and, if possible, the normalized value grouped by the selected categorical field (@id).


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.hist(filtered_df[numeric_field], bins=30, alpha=0.7)
plt.title(f"Distribution of {numeric_field} (> {threshold})")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

if group_field in filtered_df.columns and filtered_df[group_field].nunique() < 30:
    plt.figure(figsize=(12, 6))
    mean_normed = filtered_df.groupby(group_field)[normalized_col].mean()
    mean_normed.sort_values().plot(kind='bar')
    plt.title(f"Mean Normalized {numeric_field} by {group_field}")
    plt.ylabel("Mean Normalized Value")
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, filter, and explore the FAIR² dataset of mass spectrometry protein abundances from extracellular vesicles using the `mlcroissant` library. All references were made using `@id` fields for robust programmatic access. Further exploration could involve more advanced filtering by protein function or integrating additional record sets from the dataset.
